In [ ]:
from neo4j import GraphDatabase
import pickle
import networkx as nx

#### Convert .cypher KG to networkx graph and save to a .pkl file

In [ ]:
def get_nx_graph_from_neo4j(output:str,
                            url:str="bolt://localhost:7688", 
                            user:str="neo4j", 
                            passwd:str="test12345"):
    """Extract KG from Neo4j and convert it to networkx.MultiDiGraph

    Args:
        output (str): filename to save the graph in .pkl format
        url (str, optional): url to the Neo4j browser. Defaults to "bolt://localhost:7688".
        user (str, optional): user name. Defaults to "neo4j".
        passwd (str, optional): password. Defaults to "test12345".

    Returns:
        nx.MultiDiGraph: the converted graph
    """
    driver = GraphDatabase.driver(url, auth=(user, passwd))
    G = nx.MultiDiGraph() # MultiDiGraph handles multiple edges between nodes
    with driver.session() as session:
        # Match everything
        result = session.run("MATCH (n)-[r]->(m) RETURN n, r, m")
        for record in result:
            u = record['n'].element_id
            v = record['m'].element_id
            # Add nodes and edges with their properties
            G.add_node(u, **dict(record['n']))
            G.add_node(v, **dict(record['m']))
            G.add_edge(u, v, key=record['r'].element_id, type=record['r'].type, **dict(record['r']))
    
    with open(output, 'wb') as f:
        pickle.dump(G, f)
    print(f"Successfully created NetworkX graph with {len(G.nodes)} nodes!")
    return G

G = get_nx_graph_from_neo4j("./KG/healthy_aging_kg.pkl")

Successfully created NetworkX graph with 4338 nodes!


In [6]:
i = 0
for node in G.nodes(data=True):
    print(node)
    i += 1
    if i > 10:
        break

('4:53cf61c8-db19-45a6-b5ee-00baacb62768:3736', {'out_regulates': '["#821:28"]', 'in_transcribed_to': '["#940:19"]', 'involved_genes': '["Bace1"]', 'involved_other': '[]', 'in_increases': '["#690:57", "#693:57"]', 'species': 10090, 'bel': 'r(MGI:"Bace1")', 'namespace': 'MGI', 'name': 'Bace1', 'orientdb_rid': '#221:5', 'pure': True})
('4:53cf61c8-db19-45a6-b5ee-00baacb62768:1870', {'out_regulates': '["#811:0", "#812:0", "#817:0", "#826:0", "#829:0", "#810:1", "#811:1", "#812:1", "#814:1", "#815:1", "#816:1", "#825:1", "#817:3", "#818:3", "#824:3", "#824:4", "#826:32", "#821:34", "#822:34", "#817:35", "#818:35", "#819:35", "#810:41"]', 'out_increases': '["#690:7", "#690:45", "#690:64", "#690:78", "#691:7", "#691:8", "#691:45", "#692:5", "#692:7", "#692:64", "#693:7", "#694:8", "#695:64", "#696:34", "#696:77", "#697:34", "#697:65", "#698:34", "#699:65", "#700:7", "#700:44", "#700:64", "#700:67", "#701:64", "#701:66", "#701:67", "#702:44", "#702:66", "#702:67", "#704:9", "#704:26", "#704:4

In [11]:
i = 0
relations = set()
for src, dst, rel, attr in G.edges(data=True, keys=True):
    rel_type = attr.get('type')
    relations.add(rel_type)
    print(attr)
    i += 1
    if i > 10: break

{'type': 'regulates', 'annotation': '{"MESHCellularStructures": ["Extracellular Space", "Intracellular Space"], "PublicationStatus": ["Published"], "PublicationType": ["Review"], "Section": ["Full Text"], "Species": ["Mus musculus"], "mesh": ["AMP-Activated Protein Kinases", "Alzheimer Disease", "Animals", "Autophagy", "Brain", "Brain Chemistry", "Cholesterol", "Energy Metabolism", "Humans", "Mice", "Phosphorylation", "Rats", "Sphingomyelins", "TOR Serine-Threonine Kinases", "tau Proteins"], "substances": ["Sphingomyelins", "tau Proteins", "Cholesterol", "TOR Serine-Threonine Kinases", "AMP-Activated Protein Kinases"]}', 'evidence': 'Chen et al. demonstrated that metformin, at doses that lead to activation of AMPK, significantly increases the generation of both intracellular and extracellular Ab generation mediated by transcriptional up-regulation of BACE1.', 'citation': '{"full_journal_name": "Neuromolecular medicine", "pub_date": "2012-07-20", "ref": "22367557", "pub_year": 2012, "la

#### Importing .pkl triples to Neo4j database

In [ ]:
# 1. Load your KG
with open('./KG/ad_causal_only.pkl', 'rb') as f:
    kg_causal = pickle.load(f) 
with open('./KG/no_ad_causal_only.pkl', 'rb') as f:
    kg_nonad = pickle.load(f)

# 2. Connect to the specific container port
driver_causal = GraphDatabase.driver("bolt://localhost:7689", auth=("neo4j", "test1236"))
driver_nonad = GraphDatabase.driver("bolt://localhost:7690", auth=("neo4j", "test1237"))

In [25]:
for head, tail, rel, attr in kg_causal.edges(data=True, keys=True):
    print(head)
    print(tail)
    print(rel)
    for k,v in attr.items():
        print(f'{k}:, {v}')
    break

g(HGNC:"BDNF")
p(HGNC:"APP",frag("672_713"))
decreases
annotation:, {'confidence': "['Medium']", 'subgraph': "['Non-amyloidogenic subgraph', 'MAPK-ERK subgraph']"}
disease:, ad
evidence:, ["Using screening approaches in primary neurons, we identified brain- derived neurotrophic factor (BDNF) as a major inducer of Sorla that activates receptor gene transcription through the ERK (extracellular regulated kinase) pathway.These findings demonstrate that the beneficial effects ascribed to BDNF in APP metabolism act through induction of Sorla that encodes a negative regulator of neuronal APP processing"]
pmid:, [20007471]


In [16]:
for node, attr in kg_causal.nodes(data=True):
    print(node)
    print(attr)
    break

g(HGNC:"BDNF")
{'label': 'Gene', 'pure': None, 'species': 9606.0, 'name': 'BDNF', 'involved_genes': 'BDNF', 'namespace': 'HGNC', 'type': 'Gene'}


In [26]:
def add_node(tx, node_id, attrs):
    node_label = attrs.get('label', 'Entity')
    query = (
        f"MERGE (n:`{node_label}` {{id: $id}}) "
        "SET n += $props"
    )
    
    # n += $props sets all keys in the dict as properties at once
    tx.run(query, id=str(node_id), props=attrs)

def add_relationship(tx, head, tail, rel_type, attrs):
    # Match the existing nodes by ID
    query = (
        "MATCH (a {id: $head}), (b {id: $tail}) "
        f"CREATE (a)-[r:`{rel_type}`]->(b) "
        "SET r.confidence = $confidence, "
        "    r.subgraph = $subgraph, "
        "    r.evidence = $evidence,"
        "    r.disease = $disease,"
        "    r.pmid = $pmid"
    )
    
    tx.run(query, 
           head=str(head), 
           tail=str(tail), 
           confidence=attrs.get('annotation', {}).get('confidence'),
           subgraph=attrs.get('annotation', {}).get('subgraph'),
           evidence=attrs.get('evidence'),
           disease=attrs.get('disease'),
           pmid = attrs.get('pmid'))

def add_kg_to_neo4j(G:nx.MultiDiGraph,
                        url:str="bolt://localhost:7689", 
                        user:str="neo4j", 
                        passwd:str="test1236"):
    driver = GraphDatabase.driver(uri=url, auth=(user, passwd))

    with driver.session() as session:
        # add nodes:
        for node_id, attrs in G.nodes(data=True):
            session.execute_write(add_node, node_id, attrs)
        # add edges
        for head, tail, rel_type, attrs in G.edges(data=True, keys=True):
            session.execute_write(add_relationship, head, tail, rel_type, attrs)
    # close driver
    driver.close()

In [22]:
kg_causal.number_of_nodes()
kg_nonad.number_of_nodes()

3731

In [23]:
#add_kg_to_neo4j(kg_causal)
add_kg_to_neo4j(kg_nonad, "bolt://localhost:7690", "neo4j", "test1237")

### Networkx graph to Cypher
- the following code not work, need modifications 

In [ ]:
import json


def networkx_to_cypher(pkl_path, cypher_output_path, node_label="Node"):
    with open(pkl_path, "rb") as f:
        G = pickle.load(f)

    print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    print("Sample nodes:", list(G.nodes(data=True))[:2])
    print("Sample edges:", list(G.edges(data=True, keys=True))[:2])  # keys=True for MultiDiGraph

    with open(cypher_output_path, "w") as f:

        # --- Constraint ---
        f.write(":begin\n")
        f.write("CREATE CONSTRAINT UNIQUE_IMPORT_NAME FOR (node:`UNIQUE IMPORT LABEL`) REQUIRE (node.`UNIQUE IMPORT ID`) IS UNIQUE;\n")
        f.write(":commit\n")
        f.write("CALL db.awaitIndexes(300);\n\n")

        # --- Nodes in batches of 1000 ---
        nodes = list(G.nodes(data=True))
        batch_size = 1000

        for i in range(0, len(nodes), batch_size):
            batch = nodes[i:i+batch_size]
            rows = []
            for node_id, attrs in batch:
                rows.append({"_id": node_id, "properties": {k: v for k, v in attrs.items()}})

            f.write(":begin\n")
            f.write(f"UNWIND {json.dumps(rows, default=str)} AS row\n")
            f.write(f"CREATE (n:`UNIQUE IMPORT LABEL`:`{node_label}`)\n")
            f.write("SET n += row.properties\n")
            f.write("SET n.`UNIQUE IMPORT ID` = row._id;\n")
            f.write(":commit\n\n")
        
        #"CREATE (n:`UNIQUE IMPORT LABEL`{`UNIQUE IMPORT ID`: row._id}) SET n += row.properties SET n:fragment;"
        # --- Edges in batches of 1000 ---
        # MultiDiGraph edges: (src, dst, key, attrs)
        edges = list(G.edges(data=True, keys=True))

        for i in range(0, len(edges), batch_size):
            batch = edges[i:i+batch_size]
            rows = []
            for src, dst, key, attrs in batch:
                # Extract relationship type from 'rel' attr, fallback to key, then default
                rel_type = attrs.get("rel",
                           attrs.get("relation",
                           attrs.get("type",
                           attrs.get("label",
                           str(key)))))
                #print(rel_type)
                rel_type = str(rel_type).upper().replace(" ", "_").replace("-", "_")
                #print(rel_type)

                # Remaining attributes (excluding the rel type field)
                edge_props = {k: v for k, v in attrs.items() if k not in ("rel", "relation", "type", "label")}

                rows.append({
                    "src": src,
                    "dst": dst,
                    "rel": rel_type,
                    "properties": edge_props
                })

            f.write(":begin\n")
            f.write(f"UNWIND {json.dumps(rows, default=str)} AS row\n")
            f.write("MATCH (a:`UNIQUE IMPORT LABEL` {`UNIQUE IMPORT ID`: row.src})\n")
            f.write("MATCH (b:`UNIQUE IMPORT LABEL` {`UNIQUE IMPORT ID`: row.dst})\n")
            f.write("CALL apoc.merge.relationship(a, row.rel, {}, row.properties, b) YIELD rel\n")
            f.write("RETURN rel;\n")
            f.write(":commit\n\n")

        # --- Cleanup ---
        f.write(":begin\n")
        f.write("MATCH (n:`UNIQUE IMPORT LABEL`) REMOVE n:`UNIQUE IMPORT LABEL`;\n")
        f.write(":commit\n\n")
        f.write(":begin\n")
        f.write("DROP CONSTRAINT UNIQUE_IMPORT_NAME;\n")
        f.write(":commit\n")

    print(f"Done! Written to {cypher_output_path}")


# Run for your two KGs
networkx_to_cypher("./KG/ad_causal_only.pkl", "../../neo4j_containers/imports/import_causal/ad_causal.cypher", node_label="AdCausalNode")

Graph has 3732 nodes and 13133 edges
Nodes: 3732, Edges: 13133
Sample nodes: [('g(HGNC:"BDNF")', {'label': 'Gene', 'pure': None, 'species': 9606.0, 'name': 'BDNF', 'involved_genes': 'BDNF', 'namespace': 'HGNC', 'type': 'Gene'}), ('g(HGNC:"TP53")', {'label': 'Gene', 'pure': None, 'species': 9606.0, 'name': 'TP53', 'involved_genes': 'TP53', 'namespace': 'HGNC', 'type': 'Gene'})]
Sample edges: [('g(HGNC:"BDNF")', 'p(HGNC:"APP",frag("672_713"))', 'decreases', {'annotation': {'confidence': "['Medium']", 'subgraph': "['Non-amyloidogenic subgraph', 'MAPK-ERK subgraph']"}, 'disease': 'ad', 'evidence': '["Using screening approaches in primary neurons, we identified brain- derived neurotrophic factor (BDNF) as a major inducer of Sorla that activates receptor gene transcription through the ERK (extracellular regulated kinase) pathway.These findings demonstrate that the beneficial effects ascribed to BDNF in APP metabolism act through induction of Sorla that encodes a negative regulator of neurona

In [25]:
networkx_to_cypher("./KG/no_ad_causal_only.pkl", "../../neo4j_containers/imports/import_nonad/non_ad.cypher", node_label="NonAdNode")

Graph has 3731 nodes and 12110 edges
Nodes: 3731, Edges: 12110
Sample nodes: [('g(HGNC:"BDNF")', {'label': 'Gene', 'pure': None, 'species': 9606.0, 'name': 'BDNF', 'involved_genes': 'BDNF', 'namespace': 'HGNC', 'type': 'Gene'}), ('g(HGNC:"TP53")', {'label': 'Gene', 'pure': None, 'species': 9606.0, 'name': 'TP53', 'involved_genes': 'TP53', 'namespace': 'HGNC', 'type': 'Gene'})]
Sample edges: [('g(HGNC:"BDNF")', 'p(HGNC:"APP",frag("672_713"))', 'decreases', {'annotation': {'confidence': "['Medium']", 'subgraph': "['Non-amyloidogenic subgraph', 'MAPK-ERK subgraph']"}, 'disease': 'ad', 'evidence': '["Using screening approaches in primary neurons, we identified brain- derived neurotrophic factor (BDNF) as a major inducer of Sorla that activates receptor gene transcription through the ERK (extracellular regulated kinase) pathway.These findings demonstrate that the beneficial effects ascribed to BDNF in APP metabolism act through induction of Sorla that encodes a negative regulator of neurona